# 01B｜坐标系与刚体变换：从 $R p+t$ 推导到 $SE(3)$

**定位：教材级基础微课（v0.12）｜预计 100–130 分钟｜Mac CPU 即可**

前置：[01A 向量、点与坐标表示](01a_coordinate_representations.ipynb)。如果还不能
解释为什么点转换包含平移、自由向量不包含平移，请先回到 01A。

这一本回答：齐次变换究竟是什么、${}^A T_B$ 的上下标如何决定输入输出、为什么
变换组合从右向左作用、逆变换公式从哪里来，以及这些概念怎样进入 Panda 的
Cartesian 控制和 IK。

## 学习目标

完成本节后，你应该能够：

1. 区分坐标系位姿、坐标表达转换和主动移动物体；
2. 严格定义 $SE(3)$ 中的齐次刚体变换；
3. 从 ${}^A p={}^A R_B{}^B p+{}^A t_B$ 推导 4×4 形式；
4. 沿 frame 标签推导组合顺序，而不是背口诀；
5. 推导刚体逆变换中的 $-R^Tt$；
6. 用带 frame/unit/shape 注释的代码实现构造、作用、组合和求逆；
7. 在 Panda `_move_tip` 中识别位置增量、目标位姿和 IK 的边界。

## 本节知识地图

```text
点公式  p_A = R_A_B p_B + t_A_B
            ↓ 增加齐次末位 w
齐次变换 T_A_B ∈ SE(3)
            ├── 点 [p,1]：旋转 + 平移
            └── 向量 [v,0]：只旋转
            ↓ 中间 frame 标签相接
组合 T_A_C = T_A_B T_B_C
            ↓ 解方程
逆 T_B_A = T_A_B⁻¹
            ↓ 项目
tip position + increment → target pose → IK → joint control
```

## 开始前诊断

不展开答案，先选择：

1. ${}^A T_B$ 左乘 ${}^B p$ 后，结果在哪个 frame 中表达？
2. `world_from_base @ base_from_tip @ point_tip` 最先作用的是哪一项？
3. 逆变换的平移是否只是 `-t`？
4. 相同 4×4 矩阵既可能表示换坐标表达，也可能表示主动移动物体吗？

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
# 把共享教学函数目录加入 import path；这不会修改项目源码。
sys.path.insert(0, str(ROOT / "docs" / "notebooks"))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from course_feedback import check_choice, check_value, save_progress
from course_utils import assert_course_kernel

assert_course_kernel(ROOT)
print("Python kernel:", sys.executable)

output_frame_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("A", "A"), ("B", "B")],
    value=None,
    description="输出 frame",
)
first_action_quiz = widgets.RadioButtons(
    options=[
        ("请选择", None),
        ("world_from_base", "world_from_base"),
        ("base_from_tip", "base_from_tip"),
        ("point_tip", "point_tip"),
    ],
    value=None,
    description="最先作用",
)
inverse_translation_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("总是 -t", "minus_t"), ("通常是 -R.T @ t", "rotated")],
    value=None,
    description="逆平移",
)
interpretation_quiz = widgets.RadioButtons(
    options=[("请选择", None), ("可以", "yes"), ("不可以", "no")],
    value=None,
    description="两种解释",
)
display(
    output_frame_quiz,
    first_action_quiz,
    inverse_translation_quiz,
    interpretation_quiz,
)

## 严格定义

### 定义 1：刚体变换

三维刚体变换保持任意两点之间的欧氏距离和夹角。它由旋转 $R\in SO(3)$ 与平移
$t\in\mathbb{R}^3$ 组成，对点坐标作用为：

$$
p' = Rp+t.
$$

由于保持距离，刚体变换不会缩放、剪切或镜像物体。

### 定义 2：特殊欧氏群 $SE(3)$

使用齐次坐标后，刚体变换写成：

$$
T=
\begin{bmatrix}
R&t\\
0_{1\times3}&1
\end{bmatrix},
\qquad R\in SO(3),\ t\in\mathbb{R}^3.
$$

所有这样的矩阵及其矩阵乘法构成特殊欧氏群 $SE(3)$。不是任意可逆 4×4 矩阵都
属于 $SE(3)$；例如含缩放的矩阵会破坏距离。

### 定义 3：带 frame 的变换 ${}^A T_B$

${}^A T_B$ 表示“B 坐标系的位姿用 A 表达”，同时定义一个坐标转换：

$$
{}^A T_B:\ {}^B\bar p\mapsto{}^A\bar p.
$$

输入是 B 表达的齐次坐标，输出是 A 表达的齐次坐标。代码名
`a_from_b` 与该方向一致。

### 定义 4：主动与被动解释

- **被动坐标转换**：几何对象不动，只把 ${}^B p$ 改写为 ${}^A p$；
- **主动刚体运动**：坐标系固定，让物体的位置和姿态实际发生刚体运动。

两种情形可以使用数值相同的矩阵，但问题、输入语义和输出语义不同。仅看到
`T @ p` 不能判断是哪一种，必须阅读 frame 命名和上下文。

## 关键概念与符号

## 符号、单位与 shape

| 对象 | 数学记号 | shape | 单位 | 语义 |
| --- | --- | ---: | --- | --- |
| B→A 旋转 | ${}^A R_B$ | `(3,3)` | 无量纲 | B 基轴在 A 中的列表示 |
| B 原点位置 | ${}^A t_B$ | `(3,)` | m | B 原点用 A 表达 |
| B→A 刚体变换 | ${}^A T_B$ | `(4,4)` | 混合 | 输入 B，输出 A |
| 齐次点 | ${}^B\bar p=[{}^B p;1]$ | `(4,)` | 前三项 m | 会受平移 |
| 齐次向量 | ${}^B\bar v=[{}^B v;0]$ | `(4,)` | 依物理量 | 不受平移 |

4×4 矩阵内部同时保存无量纲旋转和以米为单位的平移，所以不能把整个矩阵简单说成
“单位是米”。frame 方向应进入变量名，单位优先进入点/平移变量名。

## 直观解释

可以把 ${}^A T_B$ 想成一座从“B 语言”到“A 语言”的翻译器：输入的位置数字
是用 B 的原点和尺子测得，输出数字改用 A 的原点和尺子。但这个类比不能代替计算，
因为翻译规则严格由 B 的基轴与原点在 A 中的表达决定。

frame 标签类似物理单位，能在计算前排除错误：${}^A T_B{}^B T_C$ 的 B 相接；
${}^B T_C{}^A T_B$ 的 C 与 A 不相接。与单位不同的是，错误 frame 的矩阵 shape
常常仍然兼容，因此 NumPy 不会替你发现问题。

“右边先作用”也不是视觉上的阅读习惯，而是函数复合：点先从 C 表达到 B，得到的
B 坐标才能作为 B→A 变换的输入。

## 原理与推导

### 1. 为什么齐次末位能统一点和向量

从 01A 的点转换开始：

$$
{}^A p={}^A R_B{}^B p+{}^A t_B.
$$

给点坐标追加常数 1：

$$
{}^B\bar p=\begin{bmatrix}{}^B p\\1\end{bmatrix}.
$$

进行分块矩阵乘法：

$$
\begin{bmatrix}{}^A R_B&{}^A t_B\\0&1\end{bmatrix}
\begin{bmatrix}{}^B p\\1\end{bmatrix}
=
\begin{bmatrix}{}^A R_B{}^B p+{}^A t_B\\1\end{bmatrix}.
$$

给自由向量追加 0：

$$
\begin{bmatrix}{}^A R_B&{}^A t_B\\0&1\end{bmatrix}
\begin{bmatrix}{}^B v\\0\end{bmatrix}
=
\begin{bmatrix}{}^A R_B{}^B v\\0\end{bmatrix}.
$$

平移列乘以末位 0 后消失。因此齐次末位不是“为了凑四维”，而是在矩阵乘法中编码
点和向量的不同语义。

### 2. 组合顺序从哪里来

已知 C 中的点、C 到 B、B 到 A：

$$
{}^B p={}^B R_C{}^C p+{}^B t_C,
$$

$$
{}^A p={}^A R_B{}^B p+{}^A t_B.
$$

把第一式代入第二式：

$$
{}^A p=
{}^A R_B{}^B R_C{}^C p
+{}^A R_B{}^B t_C
+{}^A t_B.
$$

所以组合后的旋转和平移分别为：

$$
{}^A R_C={}^A R_B{}^B R_C,
\qquad
{}^A t_C={}^A R_B{}^B t_C+{}^A t_B.
$$

齐次形式恰好把这两个结果统一为：

$$
{}^A T_C={}^A T_B{}^B T_C.
$$

中间标签 B 相接，最右边的 ${}^B T_C$ 最先作用于 C 坐标。这不是口诀，而是函数
复合与代入顺序。

### 3. 逆变换为什么是 $-R^Tt$

从 $p_A=Rp_B+t$ 解出 $p_B$：

$$
p_B=R^{-1}(p_A-t)=R^Tp_A-R^Tt.
$$

因此：

$$
T^{-1}=
\begin{bmatrix}
R^T&-R^Tt\\
0&1
\end{bmatrix}.
$$

逆平移通常不是 `-t`；必须先把平移向量改写到逆变换的输出 frame。

## 先预测

设 B 相对 world 旋转 90°并平移 `[2,1,0] m`，tip 相对 B 只沿 B-x 平移
`[1,0,0] m`。在运行代码前回答：

1. tip 原点在 world 中是什么？
2. `world_from_b @ b_from_tip` 与反序相同吗？
3. `inv(world_from_b)` 的平移部分是 `[-2,-1,0]` 吗？若不是，是多少？
4. `world_from_b @ b_from_tip @ tip_origin` 中的 `tip_origin` 为何最先作用？

## Worked example

${}^B t_{tip}=[1,0,0]^T$ 必须先由 ${}^{world}R_B$ 旋转到 world，再加 B 原点：

$$
{}^{world}t_{tip}
={}^{world}R_B{}^B t_{tip}+{}^{world}t_B
=[0,1,0]^T+[2,1,0]^T
=[2,2,0]^T.
$$

如果把矩阵写反，数值乘法仍可能成功，但 frame 标签变成
${}^B T_{tip}{}^{world}T_B$，相邻的 `tip` 与 `world` 无法相接。正确性首先由语义
决定，而不是由 NumPy 是否报 shape 错误决定。

## 运行与观察

下面给出最小但完整的 $SE(3)$ 实现。注释集中说明 frame、单位和数学不变量，
不逐字翻译数组赋值。

In [ ]:
def rotation_z_deg(angle_deg: float) -> np.ndarray:
  '''Returns a right-handed z rotation, shape=(3, 3), unitless.'''
  angle_rad = np.deg2rad(angle_deg)
  cosine, sine = np.cos(angle_rad), np.sin(angle_rad)
  return np.array([
      [cosine, -sine, 0.0],
      [sine, cosine, 0.0],
      [0.0, 0.0, 1.0],
  ])


def make_transform(rotation: np.ndarray, translation_m: np.ndarray) -> np.ndarray:
  '''Builds an SE(3) matrix from R and t; caller owns the frame direction.'''
  assert rotation.shape == (3, 3)
  assert translation_m.shape == (3,)
  # 单位矩阵先保证齐次最后一行严格为 [0, 0, 0, 1]。
  transform = np.eye(4)
  transform[:3, :3] = rotation       # unitless orientation block
  transform[:3, 3] = translation_m  # origin position, unit=m
  return transform


def transform_point(transform: np.ndarray, point_m: np.ndarray) -> np.ndarray:
  '''Applies a_from_b to one B-expressed point, returning A coordinates.'''
  # w=1 selects both the rotation block and the translation column.
  point_h = np.concatenate([point_m, np.array([1.0])])
  return (transform @ point_h)[:3]


def transform_vector(transform: np.ndarray, vector: np.ndarray) -> np.ndarray:
  '''Changes a vector's expression; w=0 suppresses the translation column.'''
  vector_h = np.concatenate([vector, np.array([0.0])])
  return (transform @ vector_h)[:3]


def invert_rigid_transform(a_from_b: np.ndarray) -> np.ndarray:
  '''Returns b_from_a using R.T and -R.T @ t, without a generic inverse.'''
  a_from_b_rotation = a_from_b[:3, :3]
  b_origin_a_m = a_from_b[:3, 3]
  b_from_a = np.eye(4)
  # 正交旋转的逆等于转置，这一步同时反转 frame 方向。
  b_from_a[:3, :3] = a_from_b_rotation.T
  # 原平移先取反，再由 R.T 改写到 B frame；不能只写 -t。
  b_from_a[:3, 3] = -a_from_b_rotation.T @ b_origin_a_m
  return b_from_a

In [ ]:
world_from_b = make_transform(
    rotation_z_deg(90.0),
    np.array([2.0, 1.0, 0.0]),
)
b_from_tip = make_transform(
    np.eye(3),
    np.array([1.0, 0.0, 0.0]),
)

# frame 链：world <- B <- tip。矩阵从右向左作用于 tip 坐标。
world_from_tip = world_from_b @ b_from_tip
tip_origin_tip_m = np.zeros(3)
tip_origin_world_m = transform_point(world_from_tip, tip_origin_tip_m)

b_from_world = invert_rigid_transform(world_from_b)
print("tip origin in world [m]:", np.round(tip_origin_world_m, 6))
print("inverse translation [m]:", np.round(b_from_world[:3, 3], 6))
print("world_from_b @ b_from_world:\n", np.round(world_from_b @ b_from_world, 6))

In [ ]:
# 在 world 平面画出 world、B、tip 三个原点及 frame 链。
world_origin = np.zeros(2)
b_origin_world = world_from_b[:2, 3]
tip_origin_world = tip_origin_world_m[:2]

fig, axis = plt.subplots(figsize=(7, 6))
axis.scatter(*world_origin, s=90, label="world origin")
axis.scatter(*b_origin_world, s=90, label="B origin")
axis.scatter(*tip_origin_world, s=90, label="tip origin")
axis.plot(
    [world_origin[0], b_origin_world[0], tip_origin_world[0]],
    [world_origin[1], b_origin_world[1], tip_origin_world[1]],
    "--",
    color="0.4",
    label="world <- B <- tip chain",
)
axis.set(
    xlim=(-0.5, 3.5),
    ylim=(-0.5, 3.5),
    aspect="equal",
    xlabel="world x [m]",
    ylabel="world y [m]",
    title="Frame origins after transform composition",
)
axis.legend()
axis.grid(alpha=0.25)
plt.show()

## 故意出错

下面把组合顺序写成 `b_from_tip @ world_from_b`。两个矩阵都是 4×4，所以 NumPy
不会报错；错误只能通过 frame 语义、不变量或已知几何结果发现。

运行后不要只看 `[NEXT]`。请写出错误表达式的 frame 标签，指出哪两个相邻标签
无法连接。

In [ ]:
wrong_world_from_tip = b_from_tip @ world_from_b
wrong_tip_origin_world_m = transform_point(
    wrong_world_from_tip,
    tip_origin_tip_m,
)
check_value(
    "world <- B <- tip composition",
    wrong_tip_origin_world_m,
    tip_origin_world_m,
    hint="write the frame labels; world_from_b must be left of b_from_tip",
)

## 动手修改

只改变 B 的旋转角度，保持 `b_from_tip` 的局部平移 `[1,0,0] m`。先预测：

- 角度从 0° 到 90° 时，tip 相对 B 的距离是否变化？
- tip 的 world 坐标为什么变化？
- 如果误把 `[1,0,0]` 当成 world 平移，轨迹会出现什么系统性错误？

改控件后重新运行下一格。

In [ ]:
angle_widget = widgets.SelectionSlider(
    options=[0.0, 30.0, 60.0, 90.0, 180.0],
    value=90.0,
    description="B angle°",
)
display(angle_widget)

trial_world_from_b = make_transform(
    rotation_z_deg(float(angle_widget.value)),
    np.array([2.0, 1.0, 0.0]),
)
trial_world_from_tip = trial_world_from_b @ b_from_tip
trial_tip_world_m = transform_point(trial_world_from_tip, np.zeros(3))
distance_b_to_tip_m = np.linalg.norm(
    trial_tip_world_m - trial_world_from_b[:3, 3]
)
print("tip origin in world [m]:", np.round(trial_tip_world_m, 4))
print("rigid distance B-to-tip [m]:", round(float(distance_b_to_tip_m), 6))

## 分层练习

**Level 1｜模仿。** 构造 `a_from_b`：绕 z 轴 180°、平移 `[1,2,0] m`，求
B 中点 `[0.5,0,0] m` 的 A 坐标。

**Level 2｜补全。** 不调用 `np.linalg.inv`，用 $R^T$ 和 $-R^Tt$ 写出逆变换，
并验证 `a_from_b @ b_from_a == I`。

**Level 3｜迁移。** 为 `world_from_base @ base_from_tip @ tip_from_camera` 画 frame
链，写出 `camera_from_world` 的逆链。要求每一步都从标签推出，不用“反过来乘”。

把答案写进 `notes/01b-rigid-transforms.md`。完成后再做原有综合 Notebook 和
`01_transform_3d_exercise.py`。

## 回看开始诊断

现在检查开头选择。答错时分别返回：严格定义 3、组合推导、逆变换推导、主动与
被动解释。不要把四个错误都归因于“矩阵顺序不熟”。

In [ ]:
diagnostic_checks = [
    check_choice(
        "T^A_B 的输出 frame",
        output_frame_quiz.value,
        "A",
        hint="上标是输出表达系，下标是输入表达系。",
        explanation="T^A_B 把 B 坐标改写为 A 坐标。",
    ),
    check_choice(
        "最先作用对象",
        first_action_quiz.value,
        "point_tip",
        hint="矩阵是函数复合，最右侧输入先进入。",
        explanation="point_tip 先进入 base_from_tip，再进入 world_from_base。",
    ),
    check_choice(
        "逆变换平移",
        inverse_translation_quiz.value,
        "rotated",
        hint="从 p_A=R p_B+t 解 p_B。",
        explanation="逆平移为 -R.T @ t。",
    ),
    check_choice(
        "主动与被动解释",
        interpretation_quiz.value,
        "yes",
        hint="数值矩阵不能独自决定问题语义。",
        explanation="同一矩阵可用于主动运动或被动换表达，必须看上下文。",
    ),
]

## 自测

逐条说明：它在验证 $SO(3)$、$SE(3)$、组合、逆、点/向量语义中的哪一项。

In [ ]:
# 旋转块属于 SO(3)。
rotation = world_from_b[:3, :3]
assert np.allclose(rotation.T @ rotation, np.eye(3))
assert np.isclose(np.linalg.det(rotation), 1.0)

# SE(3) 最后一行固定为 [0, 0, 0, 1]。
assert np.allclose(world_from_b[3], [0.0, 0.0, 0.0, 1.0])

# Worked example 和逆变换恒等式。
assert np.allclose(tip_origin_world_m, [2.0, 2.0, 0.0])
assert np.allclose(world_from_b @ b_from_world, np.eye(4), atol=1e-10)
assert np.allclose(b_from_world @ world_from_b, np.eye(4), atol=1e-10)

# 同一变换对点和向量的差别只来自齐次末位。
origin_b_as_point = transform_point(world_from_b, np.zeros(3))
zero_b_as_vector = transform_vector(world_from_b, np.zeros(3))
assert np.allclose(origin_b_as_point, [2.0, 1.0, 0.0])
assert np.allclose(zero_b_as_vector, [0.0, 0.0, 0.0])

# 刚体组合保持 B 到 tip 的 1 m 距离。
assert np.isclose(distance_b_to_tip_m, 1.0)
print("PASS: SE(3), composition, inverse, and homogeneous semantics")

## 项目源码连接

Panda `_move_tip` 的真实数据流是：

```text
current_tip_pos（world 中的点）
  + action[:3] * action_scale（world 中的位移向量）
  = new_tip_pos（world 中的目标点）
  + current_tip_rot
  = new_tip_mat（目标末端位姿，4×4）
  → compute_franka_ik
  → joint control
```

教学实现只讨论几何变换；真实代码还包含工作空间裁剪、IK 可解性、关节限制和后续
MuJoCo 动力学。不能因为构造了 `new_tip_mat` 就说机械臂已经移动到目标位姿。

In [ ]:
source_path = (
    ROOT
    / "mujoco_playground/_src/manipulation/franka_emika_panda/pick_cartesian.py"
)
source_lines = source_path.read_text(encoding="utf-8").splitlines()
anchors = (
    "scaled_pos =",
    "new_tip_pos = current_tip_pos",
    "new_tip_mat =",
    "compute_franka_ik(",
)

print("Source:", source_path.relative_to(ROOT))
for line_number, source_line in enumerate(source_lines, start=1):
  if any(anchor in source_line for anchor in anchors):
    print(f"{line_number:4d}: {source_line.strip()}")

## Exit ticket

先填写自己的短答案，再运行反馈格：

- `transform_direction`：${}^A T_B$ 输入、输出 frame；
- `composition`：world←base←tip 的正确代码；
- `inverse_translation`：逆变换的平移块；
- `ik_boundary`：目标 4×4 位姿是否等于物理仿真已经到达该位姿。

In [ ]:
# None 表示尚未作答。请用自己的短答案替换，不要先看反馈代码。
exit_answers = {
    "transform_direction": None,
    "composition": None,
    "inverse_translation": None,
    "ik_boundary": None,
}
print("Edit exit_answers, then run the next cell.")

In [ ]:
exit_checks = [
    check_choice(
        "变换方向",
        exit_answers["transform_direction"],
        "B to A",
        hint="下标是输入，上标是输出。",
        explanation="T^A_B maps B-expressed coordinates to A-expressed coordinates.",
    ),
    check_choice(
        "frame 链组合",
        exit_answers["composition"],
        "world_from_base @ base_from_tip",
        hint="相邻的 base 标签必须相接。",
        explanation="world <- base <- tip gives world_from_base @ base_from_tip.",
    ),
    check_choice(
        "逆平移",
        exit_answers["inverse_translation"],
        "-R.T @ t",
        hint="先把 t 旋转到逆变换的输出 frame。",
        explanation="Rigid inverse translation is -R.T @ t.",
    ),
    check_choice(
        "IK 与物理到达",
        exit_answers["ik_boundary"],
        "no",
        hint="IK 只求关节控制，后面还有动力学和接触。",
        explanation="A target pose is a command, not evidence that simulation reached it.",
    ),
]
exit_ticket_passed = all(exit_checks)

SAVE_PROGRESS = False
if SAVE_PROGRESS:
  save_progress(
      ROOT,
      "01b-rigid-transforms",
      {"se3": "green", "composition": "green", "inverse": "green"},
      exit_ticket_passed=exit_ticket_passed,
  )

## 学完请记住

关闭 Notebook 后应能脱稿说出：

1. $SE(3)$ 由合法旋转和平移组成并保持刚体距离；
2. ${}^A T_B$ 是 B frame 位姿在 A 中的表达，并把 B 坐标转换成 A 坐标；
3. 齐次点末位为 1、向量末位为 0；
4. ${}^A T_C={}^A T_B{}^B T_C$ 来自逐式代入；
5. 刚体逆的平移是 $-R^Tt$，不是一般的 `-t`；
6. 目标末端位姿、IK 关节解和动力学实际到达是三个不同阶段。

<details>
<summary>展开参考答案（完成 Exit ticket 后再看）</summary>

- ${}^A T_B$：B → A；
- `world_from_base @ base_from_tip`；
- 逆平移：`-R.T @ t`；
- 目标位姿不等于已经到达，IK 和物理仿真仍可能失败。

</details>

## 反思与记录

在 `notes/01b-rigid-transforms.md` 写下：

1. 用代入法重写一次组合公式；
2. 不看正文推导逆变换；
3. 从 `_move_tip` 标注“点 + 向量”“目标位姿”“IK 输出”三处边界；
4. 写出一个 shape 正确但 frame 语义错误的例子，以及用于发现它的断言。

然后进入[第 01 章综合交互实验](01_frames_and_transforms.ipynb)，再完成
[`01_transform_3d_exercise.py`](../labs/starter/01_transform_3d_exercise.py)。